# Human-in-the-Loop：前置動作閘門、風險分級及審計日誌

本課程的 README 以一段短程式碼介紹 Human-in-the-Loop，該程式碼會在代理已產生回應後，要求使用者進行 `APPROVE` 或 `REJECT`。這種模式是個不錯的起點，但生產環境中的 HITL 實作通常還需要三個附加部分：

1. 一個<strong>前置動作閘門</strong>，在代理執行高風險步驟<strong>之前</strong>運行，以控制成本、不可逆性和延遲。
2. <strong>風險分級</strong>，使低風險動作自動執行，中風險動作批次審核，只有高風險動作需要人類阻擋。
3. 一個<strong>審計日誌加重試循環</strong>，將每次閘門決策記錄為 JSONL，拒絕時以結構化原因重新提示代理，而不僅是打印 `Revising...`。

本筆記本在與 `06-system-message-framework.ipynb` 相同的基礎元件上構建這些功能。在 `DEMO_MODE = True` 時可以端到端執行（不需互動輸入），或在 `DEMO_MODE = False` 時以實際的 `input()` 提示執行。注意：在 DEMO_MODE 下，第三項目標的重試是以腳本方式編排，使循環機制完整可見。真正依賴修訂驅動的重新分類需要 `DEMO_MODE = False` 且有操作人員介入。

**不在範圍內（由其他課程處理）：** 身份驗證與存取控制（課程 06 README 威脅 2）、工具呼叫中介軟體（課程 14 MAF 深入探討）、多代理辯論模式。

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pattern 1: 預先動作門檻

README 的 HITL 程式碼片段先呼叫代理，然後請使用者批准輸出。這是 <strong>後動作</strong> 流程。代理已經執行，因此 LLM 呼叫費用已經支付，任何副作用（發送電郵、寫入資料庫列、發佈評論）也已經發生。

<strong>預先動作</strong> 流程會在代理執行風險步驟前插入門檻。代理提出動作，門檻決定是否執行，只有在獲批准後副作用才會發生。

| 方面 | 後動作批准（README 程式碼片段） | 預先動作門檻（本筆記本） |
|---|---|---|
| 何時執行批准？ | 代理產生輸出後 | 任何副作用執行前 |
| 拒絕時 LLM 費用 | 已經支付 | 只為提案支付，不為動作支付 |
| 不可逆副作用 | 可能（動作已經發生） | 被防止 |
| 審計清晰度 | 批准是列印語句 | 批准是帶有時間戳、動作、原因的 JSON 記錄 |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: 風險分層

並非每個操作都需要人工批准。對公共 API 的只讀查詢與發送客戶電子郵件有著不同的風險。將兩者視為相同會浪費操作員的注意力並減慢代理的運作速度。

一個簡單的三層模型：

| Tier | Examples | Approval flow |
|---|---|---|
| `low` (只讀) | 搜索知識庫，查找航班選項，獲取公共網頁 | 自動執行，記錄審計 |
| `medium` (輕量修改) | 緩存結果、計數器遞增、安排提醒 | 自動執行，但每日批次審查 |
| `high` (面向外部或不可逆) | 發送電子郵件、收費卡、發布到公共頻道 | 阻塞等待人工批准 |

這是一種分層。生產系統通常使用更細緻的層級（例如 AWS IAM 權限等級、基於角色的存取層級）。以下三層版本是混合只讀和有副作用操作的代理中最小的實用版本。

下面的分類器使用關鍵字啟發式方法，以確保演示保持確定性和低成本。在生產系統中，您會用學習型分類器或策略引擎替換它。

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Pattern 3: 審計日誌與修訂迴圈

`print("Response approved.")` 不是審計日誌。為了建立信任，每個閘道決策都應記錄為結構化事件，未來你可以查詢、重播或附加到事件回顧中。

兩部分：

1. **僅追加 JSONL。** 每個決策一行，包含時間戳、行動、層級、決策、原因。易於 grep，且日後易於發送到真正的日誌存儲。
2. **拒絕時的修訂迴圈。** 閘道回傳 `deny` 時，代理會帶著拒絕原因重新提示自己，讓下一個提案能避免問題。

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## 額外資源

其他多個公共項目實現了這些 HITL 模式的變體。比較不同的方法以找到適合你的技術棧的方案：

- **LangChain** 人類介入工具包裝器 ([docs](https://python.langchain.com/docs/integrations/tools/human_tools))：可直接使用的工具包裝器，會暫停執行以等待人類輸入。
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ 已重新架構)：使用一個 代表人類 的代理角色來處理多代理對話。
- **Semantic Kernel** 函數過濾器 ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters))：圍繞每個函數調用運行的中介軟件，適合用於門控邏輯。

每個項目對這三個子模式的處理方式不同：LangChain 將它們包裝成工具，AutoGen 使用代理角色，Semantic Kernel 使用中介過濾器。在為自己的代理選擇設計方案前，建議通讀一到兩個實作案例。

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
